In [ ]:
# Machine Learning Exercises

---

## Exercise 1: Defining the Problem and Data Collection for Loan Default Prediction

### Problem Statement
The primary prediction objective of this project is to build a binary classification model to predict whether an applicant will default on their loan (the target variable `is_default`), which will support financial analysts and underwriting systems in making informed, risk-adjusted lending decisions.

### Data Types Needed and Justification
*   **Credit Scores:** The applicant's credit score is critical because it represents their historical creditworthiness and past repayment behavior.
*   **Debt-to-Income (DTI) Ratio:** This ratio measures the applicant's monthly debt obligations relative to their gross income, helping to evaluate their capacity to take on and service new debt.
*   **Annual Income:** An applicant's annual income establishes the baseline financial inflow available to cover both living expenses and loan obligations.
*   **Employment Length:** Long-term stable employment serves as a strong indicator of steady future cash flow, reducing the risk of unexpected income disruption.
*   **Loan Amount and Interest Rate:** The structural size of the loan and its corresponding interest rate dictate the monthly burden placed on the borrower, directly affecting their likelihood of defaulting.
*   **Historical Delinquencies:** The number of past delinquencies provides concrete historical evidence of the borrower's tendencies to miss payment deadlines.

### Realistic Data Sources and Integration
*   **Internal Transaction Records:** We will query our institution's internal database (e.g., PostgreSQL or Snowflake data warehouse) using secure database connectors to extract the applicant's historical transaction logs and existing account balances.
*   **Credit Bureau Reports:** We will integrate third-party APIs (such as those provided by Equifax, Experian, or TransUnion) to retrieve credit scores, credit inquiries, and overall credit history.
*   **Employment Verification Systems:** We will utilize automated payroll verification services (like Argyle or Plaid Exchange) to directly pull and verify employment length and income records from the borrower's employer.

### Risks and Constraints
When gathering and modeling this dataset, several critical constraints must be addressed. We must comply with data privacy regulations such as GDPR and CCPA, as well as financial-specific guidelines like the Fair Credit Reporting Act (FCRA) and Equal Credit Opportunity Act (ECOA) to ensure non-discrimination. Data quality issues, such as missing fields or inaccurate records, must be systematically mitigated during ingestion. Sampling bias could easily arise if our training dataset only contains historically approved loans, creating a survival bias that fails to represent the true distribution of default risks. Finally, data governance frameworks must be strictly enforced to control data access and ensure secure storage of personally identifiable information (PII).

---

## Exercise 2: Feature Selection and Model Choice for Loan Default Prediction

```python
import pandas as pd

# This placeholder DataFrame allows the cell to run even if you did not load a dataset yet.
example_columns = [
    "age","employment_length","annual_income","credit_score","loan_amount","interest_rate",
    "debt_to_income","num_delinquencies","num_open_accounts","total_utilization","home_ownership",
    "purpose","term","application_type","state","zip_code"
]
df = pd.DataFrame(columns=example_columns)

# Selected columns based on predictive relevance
selected_features = [
    "credit_score",
    "debt_to_income",
    "annual_income",
    "loan_amount",
    "interest_rate",
    "employment_length",
    "num_delinquencies",
    "total_utilization"
]

print("Selected features:", selected_features)
```

### Feature Selection Justification
The features most relevant for predicting loan defaults are `credit_score`, `debt_to_income`, `annual_income`, `loan_amount`, `interest_rate`, `employment_length`, `num_delinquencies`, and `total_utilization`.
*   **Credit Score:** A lower credit score strongly correlates with a higher probability of default due to past credit challenges.
*   **Debt-to-Income:** A high debt-to-income ratio indicates that a substantial portion of the applicant's income is already allocated, leaving little safety margin for loan repayment.
*   **Annual Income:** Higher annual income provides a broader financial buffer to absorb unexpected expenses without defaulting on debts.
*   **Loan Amount & Interest Rate:** Higher loan values and steeper interest rates increase the monthly installment size, escalating the financial stress on the borrower.
*   **Employment Length:** Shorter employment durations imply job instability, which elevates the risk of sudden income loss.
*   **Num Delinquencies:** A high frequency of past delinquencies signals poor repayment discipline.
*   **Total Utilization:** High credit utilization indicates that the applicant is heavily reliant on revolving credit, suggesting high financial strain.

### Excluded Features
*   **Zip Code:** We exclude the `zip_code` feature because it can lead to geographic discrimination ("redlining") and violates fair lending regulations.
*   **State:** We exclude the `state` feature because localized variations are better captured by economic indices than by administrative boundaries, avoiding unnecessary high-cardinality noise.

### Encoding and Imputation
To handle categorical features like `home_ownership` and `purpose`, we will use target encoding to convert them into numeric signals while preserving predictive capacity without high-cardinality dimensionality explosion. For handling missing numeric values, we will use median imputation grouped by employment status to avoid distorting the distribution with outliers.

---

## Exercise 3: Training, Evaluating, and Optimizing the Model

### Model Selection
We select **XGBoost (Extreme Gradient Boosting)** and **Logistic Regression** as our candidate models. XGBoost is ideal because it handles non-linear relationships, interactions, and missing values exceptionally well, while Logistic Regression serves as an interpretable baseline that aligns with standard banking compliance requirements.

### Evaluation Plan
Our evaluation strategy will employ a stratified 5-fold cross-validation split to maintain the proportions of the default and non-default classes across folds. We will track and report precision, recall, F1-score, and Precision-Recall AUC (PR-AUC) rather than raw accuracy. To optimize business outcomes, the final decision threshold will be selected using a custom cost-benefit matrix that penalizes False Negatives (uninterrupted defaults) more severely than False Positives (lost business from rejected creditworthy applicants).

### Addressing Class Imbalance
To mitigate severe class imbalance, we will apply **Stratification** during splitting and adjust the **class weight** parameter (`scale_pos_weight` in XGBoost or `class_weight='balanced'` in Logistic Regression) to heavily penalize misclassified defaults.

### Hyperparameter Tuning and Data Leakage
We will use `RandomizedSearchCV` nested within our cross-validation pipeline to iteratively optimize hyperparameters like learning rate, max depth, and regularization factors. To prevent data leakage, all preprocessing steps, such as scaling and imputation, will be encapsulated inside a Scikit-Learn `Pipeline` so they are fit strictly on the training folds and never on the validation folds.

```python
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix, classification_report

# Placeholder outputs
y_true = [0, 1, 0, 1, 0, 0, 1, 0, 1, 0]
y_pred_proba = [0.05, 0.80, 0.10, 0.65, 0.20, 0.15, 0.70, 0.30, 0.85, 0.25]

# Decision threshold reflecting the business trade-off
threshold = 0.4
y_pred = [1 if p >= threshold else 0 for p in y_pred_proba]

print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_true, y_pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_true, y_pred_proba), 4))
print("PR-AUC (Average Precision):", round(average_precision_score(y_true, y_pred_proba), 4))
print("\nConfusion matrix:\n", confusion_matrix(y_true, y_pred))
```

---

## Exercise 4: Designing Machine Learning Solutions for Specific Problems

### 1. Predicting Stock Prices
*   **Paradigm:** Supervised Learning (specifically Regression / Time-Series Forecasting).
*   **Input Data:** The input data consists of historical open/high/low/close prices, trading volumes, and technical indicators over a sequential time window.
*   **Output:** The output is a continuous numerical value representing the predicted price of the stock at a specific future time interval.
*   **Learning Signal:** The learning signal is the difference (e.g., Mean Squared Error) between the model's predicted price and the actual historical price realized at that time step.

### 2. Organizing a Library of Books
*   **Paradigm:** Unsupervised Learning (specifically Clustering).
*   **Input Data:** The input data comprises high-dimensional text representations (embeddings) of book descriptions, titles, metadata, and word-frequency distributions.
*   **Output:** The output is a set of categorical cluster assignments grouping similar books together into distinct genres or topics.
*   **Learning Signal:** The objective is to minimize intra-cluster distance and maximize inter-cluster separation without explicit label supervision.

### 3. Robot Maze Navigation
*   **Paradigm:** Reinforcement Learning.
*   **Input Data:** The input data consists of real-time state observations from the robot's onboard sensors, representing its coordinates, relative distance to obstacles, and current velocity.
*   **Output:** The output is a discrete motor command or directional action chosen from the robot's action space (e.g., move forward, turn right, stop).
*   **Learning Signal:** The learning signal is a scalar reward value representing positive reinforcement for reaching the destination and negative penalties for colliding with walls or taking extra time.

---

## Exercise 5: Designing an Evaluation Strategy for Different ML Models

### Paragraph 1: Supervised Learning (Classification)
To evaluate our supervised credit default classification model, we will establish a Stratified $K$-Fold Cross-Validation setup to guarantee consistent class distribution across our evaluations. We will measure performance using metrics including Precision, Recall, F1-Score, and Receiver Operating Characteristic Area Under the Curve (ROC-AUC). ROC-AUC and Precision-Recall Curves will serve as diagnostic plots to visualize performance across all decision thresholds. A key challenge in evaluating supervised classification models is handling severe class imbalance, which can artificially inflate metrics like accuracy and hide poor performance on the minority class.

### Paragraph 2: Unsupervised Learning (Clustering)
To assess our unsupervised book clustering model, we will compute the Silhouette Coefficient to quantify how well-separated and cohesive our clusters are. We will use the Elbow Method plotted against the Within-Cluster Sum of Squares (WCSS) to determine the optimal number of clusters, alongside Davies-Bouldin index evaluations. Diagnostic plots will include t-SNE or UMAP scatter plots to visually verify cluster separation in lower dimensions. A key challenge in evaluating unsupervised learning models is the lack of ground-truth labels, making it difficult to define objective, absolute criteria for a "correct" partition.

### Paragraph 3: Reinforcement Learning
To track the success of our reinforcement learning navigation model, we will monitor the cumulative episode rewards and the step count per episode over time. We will assess convergence by analyzing if the reward curve flattens out, indicating the agent has stabilized its behavior. To manage the exploration-exploitation balance, we will implement an $\epsilon$-greedy strategy, slowly decaying the exploration probability $\epsilon$ as the training progresses to ensure final path optimization. A key challenge in evaluating reinforcement learning is the high variance and instability of training, where small changes in hyper-parameters can lead to non-convergence or catastrophic forgetting.